# 03: Evaluate Generation Quality

Цель: проверить качество ответов RAG поверх сохранённых retrieval run-файлов. Notebook читает TREC run, берёт top-k документов из `corpus.jsonl`, генерирует ответ и считает:
- bridge-метрики `gold_in_context`, `top1_is_gold`, `gold_rank`;
- reference-based метрики относительно gold answer из MedQuAD;
- reference-free groundedness/relevance;
- optional LLM-as-a-Judge.

## S3/DVC remote artifact setup

S3/DVC is the source of truth for final artifacts. Google Drive can still be used as a Colab cache, but final indexes, checkpoints, and experiment outputs should be pushed through `artifact_push`.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None


def find_local_env() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / ".env"
        if candidate.exists():
            return candidate
    return None


def load_local_env(path: str | Path | None = None):
    env_path = Path(path) if path is not None else find_local_env()
    print("Notebook cwd:", Path.cwd())
    if env_path is None or not env_path.exists():
        print("Local .env not found")
        return
    print("Loading local env from:", env_path)
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value and not os.getenv(key):
            os.environ[key] = value


# In local VS Code/Jupyter, Colab Secrets are unavailable, so read repo .env.
if userdata is None:
    load_local_env()


def set_secret(name, default=None):
    if os.getenv(name):
        return
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return
    if default is not None:
        os.environ[name] = default


set_secret("AWS_ACCESS_KEY_ID")
set_secret("AWS_SECRET_ACCESS_KEY")
set_secret("AWS_REGION", "eu-north-1")
set_secret("ARTIFACT_DVC_REMOTE", "artifact_s3")
set_secret("ARTIFACT_REMOTE_URI")

print("DVC remote:", os.getenv("ARTIFACT_DVC_REMOTE"))
print("AWS region:", os.getenv("AWS_REGION"))
print("Artifact remote URI set:", bool(os.getenv("ARTIFACT_REMOTE_URI")))

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import time

from IPython.display import display as ipy_display
import pandas as pd
import torch

REPO_OWNER = "terrylimax"
REPO_NAME = "medical-rag-reranker"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
BRANCH = "main"

RUN_ID = os.environ.get("COLAB_RUN_ID", "medquad_full_v1")
USE_DRIVE = True
MAX_EXAMPLES = 100
CONTEXT_TOP_K_VALUES = [1, 3, 5]
LLM_MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 192
GENERATION_SEED = 42

USE_LLM_JUDGE = os.environ.get("USE_LLM_JUDGE", "false").strip().lower() in {
    "1",
    "true",
    "yes",
    "y",
    "on",
}
JUDGE_BASE_URL = os.environ.get("LLM_JUDGE_BASE_URL", "http://localhost:8000/v1")
JUDGE_MODEL = os.environ.get("LLM_JUDGE_MODEL", "mistralai/Mistral-7B-Instruct-v0.3")
JUDGE_API_KEY = os.environ.get("LLM_JUDGE_API_KEY", "EMPTY")
JUDGE_MAX_CONTEXT_DOCS = int(os.environ.get("LLM_JUDGE_MAX_CONTEXT_DOCS", "5"))

PROJECT_DIR = Path("/content") / REPO_NAME
PROJECT_PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

In [ ]:
def mount_drive_or_content(use_drive: bool) -> Path:
    if not use_drive:
        return Path("/content") / "medical-rag-reranker-colab"

    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        return Path("/content/drive/MyDrive") / "medical-rag-reranker-colab"
    except Exception as exc:
        print("Drive mount failed; falling back to /content.")
        print(type(exc).__name__, str(exc))
        return Path("/content") / "medical-rag-reranker-colab"


DRIVE_BASE = mount_drive_or_content(USE_DRIVE)
DRIVE_ROOT = DRIVE_BASE / RUN_ID
MANIFEST_PATH = DRIVE_ROOT / "manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Run notebook 01 first; missing manifest: {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
RUN_ROOT = DRIVE_ROOT / "runs"
RETRIEVAL_RUN_DIR = RUN_ROOT / "retrieval"
RERANK_RUN_DIR = RUN_ROOT / "reranked"
GENERATION_ROOT = RUN_ROOT / "generation"
SUMMARY_DIR = RUN_ROOT / "summaries"
PROCESSED_DRIVE_DIR = Path(manifest["data"]["processed_drive_dir"])

for path in [GENERATION_ROOT, SUMMARY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Generation output:", GENERATION_ROOT)

In [ ]:
def sh(args, *, cwd: Path | None = None) -> None:
    args = [str(arg) for arg in args]
    print("+", " ".join(args))
    subprocess.run(args, cwd=None if cwd is None else str(cwd), check=True)


if not (PROJECT_DIR / ".git").exists():
    sh(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR])
else:
    sh(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    sh(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    sh(["git", "pull", "--ff-only"], cwd=PROJECT_DIR)

dependency_packages = [
    # Keep Colab aligned with pyproject.toml. Unpinned installs can pull
    # incompatible releases such as transformers 5.x or old datasets wheels.
    "transformers>=4.57.3,<5.0.0",
    "datasets>=4.4.2,<5.0.0",
    "tokenizers>=0.22.1,<0.23.0",
    "hydra-core>=1.3.2,<2.0.0",
    "omegaconf>=2.3.0,<3.0.0",
    "fire>=0.7.0,<0.8.0",
    "dvc>=3.0.0,<4.0.0",
    "dvc-s3>=3.3.0,<4.0.0",
    "mlflow>=3.8.1,<4.0.0",
    "torchmetrics>=1.8.2,<2.0.0",
    "rank-bm25>=0.2.2,<0.3.0",
    "sentence-transformers>=5.1.1,<6.0.0",
    "numpy>=1.26.0,<3.0.0",
    "pytrec-eval>=0.5,<0.6",
    "sqlalchemy>=2.0.45,<3.0.0",
    "alembic>=1.17.2,<2.0.0",
    # Notebook/runtime helpers.
    "pandas",
    "pyarrow",
    "scikit-learn",
    "scipy",
    "tqdm",
    "matplotlib",
    "sentencepiece",
    "accelerate>=0.26.0",
]
sh(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *dependency_packages],
    cwd=PROJECT_DIR,
)
sh(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    cwd=PROJECT_DIR,
)

if PROJECT_PROCESSED_DIR.exists():
    shutil.rmtree(PROJECT_PROCESSED_DIR)
shutil.copytree(PROCESSED_DRIVE_DIR, PROJECT_PROCESSED_DIR)
print("Restored processed data:", PROJECT_PROCESSED_DIR)

## Pull artifacts from S3/DVC

Run this after project installation so the notebook starts from the committed remote artifact state.


In [ ]:
required_env = [
    "ARTIFACT_REMOTE_URI",
    "ARTIFACT_DVC_REMOTE",
    "AWS_REGION",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
]
print("artifact_pull env:")
for name in required_env:
    value = os.getenv(name)
    if name.startswith("AWS_") and value:
        display_value = "set"
    else:
        display_value = value or "MISSING"
    print(f"  {name}: {display_value}")

missing = [
    name
    for name in ("ARTIFACT_REMOTE_URI", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
    if not os.getenv(name)
]
if missing:
    raise RuntimeError(
        "Missing Colab secrets/env vars for artifact_pull: "
        + ", ".join(missing)
        + ". Add them in Colab Secrets and enable notebook access, then rerun the setup cell."
    )

cmd = [sys.executable, "-m", "medical_rag_reranker.commands", "artifact_pull"]
print("+", " ".join(cmd))
proc = subprocess.run(cmd, text=True, capture_output=True)
if proc.stdout:
    print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f"artifact_pull failed with exit code {proc.returncode}")

## Select methods

По умолчанию берём BM25 baseline, лучший hybrid/trained вариант, graph-aware вариант и reranked вариант, если их run-файлы существуют. Список можно переопределить вручную.

In [ ]:
def existing_run(method: str) -> Path | None:
    candidates = [
        RETRIEVAL_RUN_DIR / f"{method}.trec",
        RERANK_RUN_DIR / f"{method}.trec",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


preferred_methods = [
    "bm25",
    "hybrid_minilm",
    "hybrid_trained_medcpt",
    "graph_hybrid_trained_medcpt",
    "rag_fusion_trained_medcpt",
    "hybrid_trained_medcpt__reranked",
    "graph_hybrid_trained_medcpt__reranked",
]

GENERATION_METHODS = [
    method for method in preferred_methods if existing_run(method) is not None
]
if not GENERATION_METHODS:
    raise RuntimeError("No retrieval run files found. Run notebook 02 first.")

print("Generation methods:", GENERATION_METHODS)
for method in GENERATION_METHODS:
    print(method, "->", existing_run(method))

In [ ]:
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    set_seed,
)
from medical_rag_reranker.evaluation.reference_free import (
    evaluate_generation_result,
    summarize_generation_evaluations,
)
from medical_rag_reranker.evaluation.llm_judge import LocalOpenAICompatibleJudge

set_seed(GENERATION_SEED)


def read_jsonl(path: Path, limit: int | None = None) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_docstore(corpus_path: Path) -> dict[str, dict]:
    rows = read_jsonl(corpus_path)
    return {str(row["doc_id"]): row for row in rows if row.get("doc_id")}


def read_qrels(path: Path) -> dict[str, list[str]]:
    qrels: dict[str, list[str]] = {}
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            qid, _iter, doc_id, rel = line.split()[:4]
            if int(rel) > 0:
                qrels.setdefault(str(qid), []).append(str(doc_id))
    return qrels


def read_trec_run(path: Path) -> dict[str, list[tuple[str, float]]]:
    runs: dict[str, list[tuple[str, float]]] = {}
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            qid, _q0, doc_id, _rank, score, _run_name = line.split()[:6]
            runs.setdefault(qid, []).append((doc_id, float(score)))
    for qid in list(runs):
        runs[qid].sort(key=lambda item: item[1], reverse=True)
    return runs


def query_id(row: dict, idx: int) -> str:
    return str(row.get("query_id") or row.get("question_id") or f"query-{idx}")


def query_text(row: dict) -> str:
    value = row.get("question") or row.get("text")
    if not value:
        raise ValueError("Query row must contain question or text")
    return str(value)


def tokenize_reference(text: str) -> list[str]:
    normalized = "".join(ch.lower() if ch.isalnum() else " " for ch in str(text or ""))
    return [token for token in normalized.split() if token]


def lcs_len(left: list[str], right: list[str]) -> int:
    if not left or not right:
        return 0
    prev = [0] * (len(right) + 1)
    for token_left in left:
        curr = [0] * (len(right) + 1)
        for j, token_right in enumerate(right, start=1):
            if token_left == token_right:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(prev[j], curr[j - 1])
        prev = curr
    return prev[-1]


def rouge_l_f1(candidate: str, reference: str) -> float:
    cand = tokenize_reference(candidate)
    ref = tokenize_reference(reference)
    if not cand or not ref:
        return 0.0
    lcs = lcs_len(cand, ref)
    precision = lcs / float(len(cand))
    recall = lcs / float(len(ref))
    if precision + recall == 0.0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def lexical_cosine(candidate: str, reference: str) -> float:
    from collections import Counter
    import math

    cand = Counter(tokenize_reference(candidate))
    ref = Counter(tokenize_reference(reference))
    if not cand or not ref:
        return 0.0
    shared = set(cand) & set(ref)
    dot = sum(cand[token] * ref[token] for token in shared)
    cand_norm = math.sqrt(sum(value * value for value in cand.values()))
    ref_norm = math.sqrt(sum(value * value for value in ref.values()))
    return dot / float(cand_norm * ref_norm + 1e-12)


def find_rank(
    ranked_docs: list[tuple[str, float]], gold_doc_ids: set[str]
) -> int | None:
    for rank, (doc_id, _score) in enumerate(ranked_docs, start=1):
        if doc_id in gold_doc_ids:
            return rank
    return None


def read_latency_profile(run_file: Path) -> list[float]:
    latency_path = run_file.with_suffix(run_file.suffix + ".latency.json")
    if not latency_path.exists():
        return []
    payload = json.loads(latency_path.read_text(encoding="utf-8"))
    return [float(value) for value in payload.get("latencies_ms", [])]


def build_prompt(question: str, docs: list[dict]) -> str:
    if docs:
        context = "\n\n".join(f"[{doc['doc_id']}] {doc['text']}" for doc in docs)
    else:
        context = "(no retrieved documents)"
    return (
        "You are a medical QA assistant.\n"
        "Rules:\n"
        "1) Answer strictly using only the provided context.\n"
        "2) If context is insufficient, say so explicitly.\n"
        "3) Cite supporting sources using [doc_id] format.\n"
        "4) Do not invent citations.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

In [ ]:
class NotebookGenerator:
    def __init__(self, model_name: str) -> None:
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.is_seq2seq = True
        try:
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        except Exception:
            self.is_seq2seq = False
            self.model = AutoModelForCausalLM.from_pretrained(model_name)
        if (
            self.tokenizer.pad_token_id is None
            and self.tokenizer.eos_token_id is not None
        ):
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.to(self.device)
        self.model.eval()

    def generate(self, prompt: str) -> str:
        encoded = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(self.device)
        kwargs = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": False,
            "pad_token_id": self.tokenizer.pad_token_id,
        }
        with torch.no_grad():
            output = self.model.generate(**encoded, **kwargs)
        if self.is_seq2seq:
            return self.tokenizer.decode(output[0], skip_special_tokens=True).strip()
        prompt_len = encoded["input_ids"].shape[1]
        return self.tokenizer.decode(
            output[0][prompt_len:], skip_special_tokens=True
        ).strip()


generator = NotebookGenerator(LLM_MODEL_NAME)
print("Loaded generator:", LLM_MODEL_NAME, "on", generator.device)

judge = None
if USE_LLM_JUDGE:
    judge = LocalOpenAICompatibleJudge(
        base_url=JUDGE_BASE_URL,
        model=JUDGE_MODEL,
        api_key=JUDGE_API_KEY,
        temperature=0.0,
        timeout_seconds=60.0,
        max_tokens=512,
        max_context_docs=JUDGE_MAX_CONTEXT_DOCS,
        max_doc_chars=1500,
    )
    print("LLM-as-a-Judge enabled:", JUDGE_BASE_URL, JUDGE_MODEL)
else:
    print("LLM-as-a-Judge disabled. Set USE_LLM_JUDGE=true to enable it.")

In [ ]:
CITATION_PATTERN = re.compile(r"\[([^\[\]]+)\]")


def detect_citations(answer: str) -> list[str]:
    seen = set()
    out = []
    for raw in CITATION_PATTERN.findall(answer):
        citation = raw.strip()
        if citation and citation not in seen:
            seen.add(citation)
            out.append(citation)
    return out


def summarize_gold_aware_rows(rows: list[dict]) -> dict:
    if not rows:
        raise ValueError("Cannot summarize empty generation rows.")

    summary = summarize_generation_evaluations(rows)
    numeric_keys = [
        "gold_in_context",
        "top1_is_gold",
        "citation_points_to_gold",
        "rouge_l_f1_to_gold",
        "lexical_cosine_to_gold",
        "gold_rank_found",
        "reciprocal_gold_rank",
        "gold_in_top_1",
        "gold_in_top_3",
        "gold_in_top_5",
        "gold_in_top_10",
    ]
    for key in numeric_keys:
        values = [float(row.get(key, 0.0)) for row in rows]
        summary[f"avg_{key}"] = sum(values) / float(max(1, len(values)))

    ranks = [int(row["gold_rank"]) for row in rows if row.get("gold_rank") is not None]
    summary["gold_rank_not_found_rate"] = 1.0 - summary["avg_gold_rank_found"]
    if ranks:
        ordered = sorted(ranks)
        mid = len(ordered) // 2
        summary["gold_rank_mean"] = sum(ordered) / float(len(ordered))
        summary["gold_rank_median"] = (
            float(ordered[mid])
            if len(ordered) % 2
            else (ordered[mid - 1] + ordered[mid]) / 2.0
        )
    else:
        summary["gold_rank_mean"] = 0.0
        summary["gold_rank_median"] = 0.0

    judged = [row for row in rows if isinstance(row.get("llm_judge"), dict)]
    summary["llm_judge_coverage_rate"] = len(judged) / float(max(1, len(rows)))
    summary["llm_judge_error_rate"] = sum(
        1 for row in rows if row.get("llm_judge_error")
    ) / float(max(1, len(rows)))
    for key in ["faithfulness", "relevance", "completeness", "safety"]:
        values = [
            float(row["llm_judge"][key]) for row in judged if key in row["llm_judge"]
        ]
        if values:
            summary[f"avg_judge_{key}"] = sum(values) / float(len(values))
    verdicts = [
        str(row["llm_judge"].get("verdict", "")).strip().lower()
        for row in judged
        if row["llm_judge"].get("verdict")
    ]
    if verdicts:
        summary["judge_pass_rate"] = sum(
            1 for verdict in verdicts if verdict == "pass"
        ) / float(len(verdicts))
    return summary


def generate_for_run(
    method: str, run_file: Path, context_top_k: int, judge=None
) -> dict:
    queries = read_jsonl(
        PROJECT_PROCESSED_DIR / "eval_queries.jsonl", limit=MAX_EXAMPLES
    )
    docstore = load_docstore(PROJECT_PROCESSED_DIR / "corpus.jsonl")
    qrels = read_qrels(PROJECT_PROCESSED_DIR / "qrels.tsv")
    run = read_trec_run(run_file)
    latencies_ms = read_latency_profile(run_file)

    rows = []
    for idx, query_row in enumerate(queries, start=1):
        qid = query_id(query_row, idx)
        question = query_text(query_row)
        ranked_for_query = run.get(qid, [])
        ranked_docs = ranked_for_query[: int(context_top_k)]
        gold_doc_ids = set(qrels.get(qid, []))
        explicit_gold = query_row.get("gold_doc_id")
        if explicit_gold:
            gold_doc_ids.add(str(explicit_gold))
        gold_doc_id = next(iter(sorted(gold_doc_ids)), None)
        gold_doc = docstore.get(gold_doc_id, {}) if gold_doc_id else {}
        reference_answer = str(gold_doc.get("text") or "")
        gold_rank = find_rank(ranked_for_query, gold_doc_ids)

        docs = []
        for doc_id, score in ranked_docs:
            doc = docstore.get(doc_id, {})
            docs.append(
                {
                    "doc_id": doc_id,
                    "score": float(score),
                    "text": str(doc.get("text") or ""),
                    "source": doc.get("source"),
                }
            )

        prompt = build_prompt(question, docs)
        started = time.perf_counter()
        answer = generator.generate(prompt)
        generation_latency_ms = (time.perf_counter() - started) * 1000.0
        citations = detect_citations(answer)
        retrieved_doc_ids = {doc["doc_id"] for doc in docs}
        retrieval_latency_ms = (
            float(latencies_ms[idx - 1]) if idx - 1 < len(latencies_ms) else 0.0
        )
        gold_in_context = bool(gold_doc_ids & retrieved_doc_ids)
        top1_doc_id = docs[0]["doc_id"] if docs else None

        row = {
            "query_id": qid,
            "question": question,
            "method": method,
            "run_file": str(run_file),
            "context_top_k": int(context_top_k),
            "gold_doc_id": gold_doc_id,
            "gold_doc_ids": sorted(gold_doc_ids),
            "gold_rank": gold_rank,
            "reference_answer": reference_answer,
            "top1_doc_id": top1_doc_id,
            "top1_is_gold": float(top1_doc_id in gold_doc_ids) if top1_doc_id else 0.0,
            "gold_in_context": float(gold_in_context),
            "gold_rank_found": float(gold_rank is not None),
            "reciprocal_gold_rank": 0.0
            if gold_rank is None
            else 1.0 / float(gold_rank),
            "gold_in_top_1": float(gold_rank is not None and gold_rank <= 1),
            "gold_in_top_3": float(gold_rank is not None and gold_rank <= 3),
            "gold_in_top_5": float(gold_rank is not None and gold_rank <= 5),
            "gold_in_top_10": float(gold_rank is not None and gold_rank <= 10),
            "retrieved": docs,
            "answer": answer,
            "citations_detected": citations,
            "supported_citations_detected": [
                c for c in citations if c in retrieved_doc_ids
            ],
            "unsupported_citations_detected": [
                c for c in citations if c not in retrieved_doc_ids
            ],
            "citation_points_to_gold": float(any(c in gold_doc_ids for c in citations)),
            "rouge_l_f1_to_gold": rouge_l_f1(answer, reference_answer),
            "lexical_cosine_to_gold": lexical_cosine(answer, reference_answer),
            "retrieval_latency_ms": retrieval_latency_ms,
            "generation_latency_ms": float(generation_latency_ms),
            "end_to_end_latency_ms": float(
                retrieval_latency_ms + generation_latency_ms
            ),
            "reranker_enabled": "reranked" in method,
        }
        row["evaluation"] = evaluate_generation_result(row)
        if judge is not None:
            try:
                row["llm_judge"] = judge.evaluate(row)
            except Exception as exc:
                row["llm_judge_error"] = f"{type(exc).__name__}: {exc}"
        rows.append(row)

    summary = summarize_gold_aware_rows(rows)
    summary.update(
        {
            "method": method,
            "run_file": str(run_file),
            "llm_model_name": LLM_MODEL_NAME,
            "context_top_k": int(context_top_k),
            "max_examples": MAX_EXAMPLES,
            "use_llm_judge": bool(judge is not None),
            "judge_base_url": JUDGE_BASE_URL if judge is not None else None,
            "judge_model": JUDGE_MODEL if judge is not None else None,
            "generation_latency_mean_ms": sum(
                row["generation_latency_ms"] for row in rows
            )
            / max(1, len(rows)),
        }
    )
    return {"rows": rows, "summary": summary}

In [ ]:
def truncate(text: str, max_chars: int = 240) -> str:
    clean = " ".join(str(text or "").split())
    if len(clean) <= max_chars:
        return clean
    return clean[: max_chars - 3] + "..."


def write_generation_report(path: Path, summary: dict, rows: list[dict]) -> None:
    lines = [
        f"# Generation Evaluation: {summary['method']}",
        "",
        "## Summary",
        "",
    ]
    for key in sorted(summary):
        value = summary[key]
        if isinstance(value, float):
            lines.append(f"- `{key}`: {value:.4f}")
        else:
            lines.append(f"- `{key}`: {value}")
    lines.append("")
    lines.append("## Examples")
    lines.append("")
    for idx, row in enumerate(rows, start=1):
        lines.append(f"### Example {idx} (`{row['query_id']}`)")
        lines.append("")
        lines.append(f"**Question**: {row['question']}")
        lines.append("")
        ev = row["evaluation"]
        lines.append(
            f"**Scores**: context_relevance={ev['context_relevance']:.3f}, "
            f"groundedness={ev['groundedness']:.3f}, "
            f"answer_relevance={ev['answer_relevance']:.3f}"
        )
        lines.append(
            f"**Gold**: gold_doc_id=`{row.get('gold_doc_id')}`, "
            f"gold_rank={row.get('gold_rank')}, "
            f"gold_in_context={row.get('gold_in_context')}, "
            f"rouge_l_f1={row.get('rouge_l_f1_to_gold', 0.0):.3f}, "
            f"lexical_cosine={row.get('lexical_cosine_to_gold', 0.0):.3f}"
        )
        if isinstance(row.get("llm_judge"), dict):
            judge_row = row["llm_judge"]
            lines.append(
                "**LLM judge**: "
                f"faithfulness={float(judge_row.get('faithfulness', 0.0)):.1f}, "
                f"relevance={float(judge_row.get('relevance', 0.0)):.1f}, "
                f"completeness={float(judge_row.get('completeness', 0.0)):.1f}, "
                f"safety={float(judge_row.get('safety', 0.0)):.1f}, "
                f"verdict={judge_row.get('verdict')}"
            )
        elif row.get("llm_judge_error"):
            lines.append(f"**LLM judge error**: {row['llm_judge_error']}")
        lines.append("")
        lines.append("**Top docs**:")
        for rank, doc in enumerate(row["retrieved"], start=1):
            marker = (
                " [GOLD]" if doc["doc_id"] in set(row.get("gold_doc_ids") or []) else ""
            )
            lines.append(
                f"{rank}. `{doc['doc_id']}`{marker} (score={doc['score']:.4f}) - {truncate(doc['text'])}"
            )
        lines.append("")
        lines.append("**Answer:**")
        lines.append("")
        lines.append(row["answer"])
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")


generation_summaries = []
for context_top_k in CONTEXT_TOP_K_VALUES:
    for method in GENERATION_METHODS:
        run_file = existing_run(method)
        print(f"\n== Generate {method} | context_top_k={context_top_k} ==")
        result = generate_for_run(
            method,
            run_file,
            context_top_k=context_top_k,
            judge=judge,
        )
        out_dir = GENERATION_ROOT / method / f"topk_{context_top_k}"
        out_dir.mkdir(parents=True, exist_ok=True)

        raw_jsonl = out_dir / "generation_eval.jsonl"
        summary_json = out_dir / "summary.json"
        report_md = out_dir / "report.md"

        write_jsonl(raw_jsonl, result["rows"])
        summary_json.write_text(
            json.dumps(result["summary"], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        write_generation_report(report_md, result["summary"], result["rows"])
        generation_summaries.append(result["summary"])

        print("Saved:", raw_jsonl)
        print("Saved:", summary_json)
        print("Saved:", report_md)

gen_df = pd.DataFrame(generation_summaries)
comparison_csv = SUMMARY_DIR / "generation_quality_summary.csv"
comparison_json = SUMMARY_DIR / "generation_quality_summary.json"
gen_df.to_csv(comparison_csv, index=False)
gen_df.to_json(comparison_json, orient="records", force_ascii=False, indent=2)

print("Saved generation comparison:", comparison_csv)
sort_key = (
    "avg_rouge_l_f1_to_gold"
    if "avg_rouge_l_f1_to_gold" in gen_df.columns
    else "avg_groundedness"
)
ipy_display(
    gen_df.sort_values(sort_key, ascending=False)
    if sort_key in gen_df.columns
    else gen_df
)